# 07 | Expanded Subject-Level Development Evaluation

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Objective:** Expand the development cohort from six to ten HARTH v2.0 participants, audit documented-class coverage, and repeat participant-disjoint cross-validation. **Keep the original S023/S032 final Test participants completely excluded from data loading, feature construction, training, CV, and model selection in this notebook.**

**Research status:** Exploratory development study, not final held-out test evaluation or a full 31-subject benchmark. The additional four participants are S029, S030, S031 and S033; source files are downloaded only if not already cached.

## 01. Setup and reusable functions

The shared loaders, windowing, feature extraction and model evaluation stay in `.py` files. Code cells contain short English comments explaining their actions.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Locate the repository root, then import the existing Python functions.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root == root.parent:
        raise RuntimeError("Run from within the human-movement-intelligence repository.")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))

from harth_data import ACTIVITY_NAMES, prepare_harth_feature_cohort
from core.group_cross_validation import evaluate_group_cross_validation

print("Shared research components: ready")

## 02. Freeze the final Test IDs and expand the development cohort

The two Test participants from Notebook 05/06 are **not loaded** in this notebook. We reuse six existing development participants and add four new ones.

In [ ]:
# Keep the original Test participants out of every development operation.
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = [
    "S024", "S025", "S027", "S028", "S034", "S035",
    "S029", "S030", "S031", "S033",
]
TASK_CLASSES = tuple(range(1, 9))
assert len(DEVELOPMENT_SUBJECTS) == len(set(DEVELOPMENT_SUBJECTS)) == 10
assert not (set(DEVELOPMENT_SUBJECTS) & RESERVED_TEST)

# Build per-participant windows and deterministic statistical features.
features, metadata, overview, excluded = prepare_harth_feature_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES,
    window_size=100, step_size=50, sampling_rate=50,
)
display(overview)
print("Expanded development matrix:", features.shape)
print("Original final Test participants loaded: NO")

## 03. Review label provenance and activity coverage

The restricted task contains only documented codes 1–8. All other codes are counted as excluded **after time-safe windowing**. Inspect which classes occur for each person.

In [ ]:
# Display excluded source labels rather than silently discarding their evidence.
print("Excluded windows by original source code:")
display(excluded.sort_values(["source_code", "subject_id"]))

# Count eligible windows for each documented class and development subject.
class_coverage = pd.crosstab(
    metadata["subject_id"], metadata["label"]
).reindex(index=DEVELOPMENT_SUBJECTS, columns=TASK_CLASSES, fill_value=0)
class_coverage.columns = [
    f"{code}: {ACTIVITY_NAMES[code]}" for code in TASK_CLASSES
]
display(class_coverage)
print("Classes present per subject:", (class_coverage > 0).sum(axis=1).to_dict())
print("Subjects per class:", (class_coverage > 0).sum(axis=0).to_dict())

## 04. Visualise subject-level class availability

A heatmap shows how unequal activity coverage is across participants. It is a count map, not a performance chart.

In [ ]:
# Plot the per-subject class window counts; annotate absent classes explicitly.
fig, ax = plt.subplots(figsize=(13, 5.5))
heat = ax.imshow(class_coverage.to_numpy(), aspect="auto", cmap="YlGnBu")
ax.set_xticks(np.arange(len(class_coverage.columns)), class_coverage.columns, rotation=55, ha="right")
ax.set_yticks(np.arange(len(class_coverage.index)), class_coverage.index)
ax.set(title="HARTH expanded development cohort: windows per activity and subject",
       xlabel="Documented activity", ylabel="Development subject")
fig.colorbar(heat, ax=ax, label="Valid windows")
fig.tight_layout()
plt.show()

## 05. Run grouped CV with class-coverage safeguards

Five folds leave out two development participants each. Every model is refitted inside each fold; `StandardScaler` learns from fold Train only. The generic CV function checks for training-class coverage and will stop explicitly if a fold is scientifically unsuitable.

In [ ]:
# Evaluate only the ten development subjects, never the original final Test.
fold_results, subject_results, summary = evaluate_group_cross_validation(
    features=features.to_numpy(),
    labels=metadata["label"].to_numpy(),
    group_ids=metadata["subject_id"].to_numpy(),
    classes=TASK_CLASSES,
    n_splits=5,
    random_state=42,
)
display(summary.round(4))
display(fold_results[[
    "fold", "model", "train_subjects", "validation_subjects",
    "validation_classes_present", "macro_f1", "balanced_accuracy"
]].round(4))

## 06. Compare held-out-fold and subject-level model stability

Fold means can disguise subject differences. Show both fold-level fixed-task Macro F1 and performance across individual held-out participants.

In [ ]:
# Plot model-specific held-out-fold scores.
fig, axes = plt.subplots(2, 1, figsize=(11, 9))
for name, part in fold_results.groupby("model", sort=False):
    part = part.sort_values("fold")
    axes[0].plot(part["fold"], part["macro_f1"], marker="o", label=name)
axes[0].set(title="Expanded development: held-out-fold Macro F1",
            xlabel="CV fold", ylabel="Macro F1", ylim=(0, 1))
axes[0].grid(alpha=0.25)
axes[0].legend()

# Plot fixed eight-class Macro F1 for each participant.
per_subject = subject_results.pivot(
    index="subject_id", columns="model", values="macro_f1_fixed_task"
).sort_index()
per_subject.plot(kind="bar", ax=axes[1], rot=0)
axes[1].set(title="Held-out-participant Macro F1 (eight-class denominator)",
            xlabel="Participant", ylabel="Macro F1", ylim=(0, 1))
axes[1].grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()
display(subject_results.sort_values(["model", "subject_id"]).round(4))

## 07. Scientific quality gate

This notebook expands development evidence while preserving the final Test cohort. It does **not** claim robust external generalisation, compare against published HARTH benchmarks, or produce final Test metrics.

In [ ]:
# Check complete subject coverage and strict exclusion of reserved Test IDs.
reported_subjects = set(subject_results["subject_id"])
assert reported_subjects == set(DEVELOPMENT_SUBJECTS)
assert not (reported_subjects & RESERVED_TEST)
assert subject_results.groupby(["model", "subject_id"]).size().eq(1).all()
assert fold_results["macro_f1"].between(0, 1).all()
assert len(fold_results) == 5 * 3
assert np.isfinite(features.to_numpy()).all()

print("Expanded development grouped CV: PASSED")
print("Original final Test data loaded/evaluated: NO")
print("Clinical/generalisation claim: NOT SUPPORTED")